# 🤝 Exercícios — Sistemas Multiagentes

**Disciplina:** Inteligência Artificial | **Nível:** Intermediário

> Simule agentes autônomos que interagem, cooperam e competem em ambientes compartilhados.


## 1. Agentes Simples em um Ambiente

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

random.seed(42)

class Agente:
    def __init__(self, nome, x, y, energia=100):
        self.nome = nome
        self.x = x
        self.y = y
        self.energia = energia
        self.historico = [(x, y)]
    
    def perceber(self, ambiente):
        """Percebe recursos na posição atual."""
        return ambiente.get((self.x, self.y), 0)
    
    def agir(self, ambiente, tamanho_grid=20):
        """Movimento aleatório com consumo de energia."""
        recurso = self.perceber(ambiente)
        if recurso > 0:
            self.energia += recurso
            ambiente[(self.x, self.y)] = 0
        # Move aleatoriamente
        dx, dy = random.choice([(0,1),(0,-1),(1,0),(-1,0)])
        self.x = max(0, min(tamanho_grid-1, self.x + dx))
        self.y = max(0, min(tamanho_grid-1, self.y + dy))
        self.energia -= 1  # custo de movimento
        self.historico.append((self.x, self.y))
        return self.energia > 0  # sobreviveu?

class Ambiente:
    def __init__(self, tamanho=20, n_recursos=50):
        self.tamanho = tamanho
        self.recursos = {}
        for _ in range(n_recursos):
            x, y = random.randint(0, tamanho-1), random.randint(0, tamanho-1)
            self.recursos[(x,y)] = random.randint(5,20)
    
    def recarregar(self, taxa=0.1):
        """Adiciona novos recursos aleatoriamente."""
        if random.random() < taxa:
            x, y = random.randint(0,self.tamanho-1), random.randint(0,self.tamanho-1)
            self.recursos[(x,y)] = random.randint(5,15)

# Simulação
ambiente = Ambiente(tamanho=20, n_recursos=60)
agentes = [Agente(f"A{i}", random.randint(0,19), random.randint(0,19)) for i in range(5)]

energias = {a.nome: [a.energia] for a in agentes}

for passo in range(200):
    ambiente.recarregar()
    agentes_vivos = [a for a in agentes if a.agir(ambiente.recursos)]
    for a in agentes_vivos:
        energias[a.nome].append(a.energia)

# Visualização das energias
plt.figure(figsize=(10,4))
for nome, hist in energias.items():
    plt.plot(hist, label=nome)
plt.xlabel('Passo'); plt.ylabel('Energia')
plt.title('Evolução da Energia dos Agentes'); plt.legend(); plt.grid(True); plt.show()

# Trajetória dos agentes
plt.figure(figsize=(8,8))
for a in agentes:
    xs, ys = zip(*a.historico)
    plt.plot(xs, ys, alpha=0.6, label=a.nome)
    plt.scatter([a.x],[a.y],s=80,zorder=5)
plt.title('Trajetórias dos Agentes'); plt.legend(); plt.grid(True); plt.show()


### 📝 Exercício 1

Modifique a estratégia de movimento dos agentes para que eles se movam **em direção ao recurso mais próximo** em vez de aleatoriamente. Compare as energias finais com a versão aleatória.

In [ ]:
class AgenteInteligente(Agente):
    """Agente que busca recursos próximos."""
    
    def agir(self, ambiente, tamanho_grid=20):
        recurso = self.perceber(ambiente)
        if recurso > 0:
            self.energia += recurso
            ambiente[(self.x, self.y)] = 0
        
        # TODO: mova em direção ao recurso mais próximo
        # Dica: itere sobre ambiente.items() e calcule distância Manhattan
        
        # Por ora, movimento aleatório como fallback:
        dx, dy = random.choice([(0,1),(0,-1),(1,0),(-1,0)])
        self.x = max(0, min(tamanho_grid-1, self.x + dx))
        self.y = max(0, min(tamanho_grid-1, self.y + dy))
        self.energia -= 1
        self.historico.append((self.x, self.y))
        return self.energia > 0

random.seed(42)
amb2 = Ambiente(tamanho=20, n_recursos=60)
agentes2 = [AgenteInteligente(f"AI{i}", random.randint(0,19), random.randint(0,19)) for i in range(5)]
energias2 = {a.nome: [a.energia] for a in agentes2}
for _ in range(200):
    amb2.recarregar()
    for a in agentes2:
        if a.agir(amb2.recursos):
            energias2[a.nome].append(a.energia)
print("Energia final dos agentes inteligentes:")
for a in agentes2: print(f"  {a.nome}: {a.energia}")


## 2. Emergência — O Modelo de Schelling

O modelo de Schelling mostra como preferências individuais leves podem gerar **segregação** emergente.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

def criar_grade(tamanho=20, proporcao_vazia=0.2):
    """0=vazio, 1=tipo A, 2=tipo B"""
    grade = np.zeros((tamanho, tamanho), dtype=int)
    for i in range(tamanho):
        for j in range(tamanho):
            r = random.random()
            if r < 0.4: grade[i,j] = 1
            elif r < 0.8: grade[i,j] = 2
    return grade

def calcular_satisfacao(grade, i, j, limiar=0.3):
    """Agente satisfeito se >= limiar% dos vizinhos são do mesmo tipo."""
    tipo = grade[i,j]
    if tipo == 0: return True
    vizinhos = []
    for di in [-1,0,1]:
        for dj in [-1,0,1]:
            if di==0 and dj==0: continue
            ni,nj = (i+di)%len(grade), (j+dj)%len(grade[0])
            if grade[ni,nj] != 0:
                vizinhos.append(grade[ni,nj]==tipo)
    return (sum(vizinhos)/len(vizinhos) >= limiar) if vizinhos else True

def simular_schelling(grade, limiar=0.3, n_passos=5):
    grade = grade.copy()
    for _ in range(n_passos):
        insatisfeitos = [(i,j) for i in range(len(grade)) for j in range(len(grade[0]))
                        if grade[i,j]!=0 and not calcular_satisfacao(grade,i,j,limiar)]
        vazios = [(i,j) for i in range(len(grade)) for j in range(len(grade[0])) if grade[i,j]==0]
        for (i,j) in insatisfeitos:
            if not vazios: break
            ni,nj = random.choice(vazios)
            grade[ni,nj] = grade[i,j]
            grade[i,j] = 0
            vazios.remove((ni,nj))
            vazios.append((i,j))
    return grade

random.seed(42)
grade_inicial = criar_grade(20)
grade_final = simular_schelling(grade_inicial, limiar=0.3, n_passos=20)

fig, axes = plt.subplots(1,2,figsize=(12,5))
for ax, g, titulo in zip(axes, [grade_inicial, grade_final], ['Estado Inicial','Após Segregação (20 passos)']):
    ax.imshow(g, cmap='RdBu', vmin=0, vmax=2)
    ax.set_title(titulo); ax.axis('off')
plt.suptitle('Modelo de Schelling — Emergência de Segregação', fontsize=13)
plt.tight_layout(); plt.show()


### 📝 Exercício 2

Experimente com diferentes valores de `limiar` (0.1, 0.3, 0.5, 0.7). Para qual valor a segregação é mais intensa? Existe um limiar abaixo do qual a segregação não ocorre?

In [ ]:
limiares = [0.1, 0.3, 0.5, 0.7]
fig, axes = plt.subplots(1, len(limiares), figsize=(16,4))
for ax, lim in zip(axes, limiares):
    random.seed(42)
    g0 = criar_grade(20)
    gf = simular_schelling(g0, limiar=lim, n_passos=20)
    ax.imshow(gf, cmap='RdBu', vmin=0, vmax=2)
    ax.set_title(f'Limiar={lim}'); ax.axis('off')
plt.suptitle('Impacto do Limiar no Modelo de Schelling')
plt.tight_layout(); plt.show()


## 3. Cooperação e Dilema do Prisioneiro

In [ ]:
# Dilema do Prisioneiro Iterado entre agentes
import random

PAYOFF = {
    ('C','C'): (3,3),   # ambos cooperam
    ('C','D'): (0,5),   # eu coopero, ele trai
    ('D','C'): (5,0),   # eu traio, ele coopera
    ('D','D'): (1,1),   # ambos traem
}

class AgenteDP:
    def __init__(self, nome, estrategia):
        self.nome = nome
        self.estrategia = estrategia
        self.pontos = 0
        self.historico = []
    
    def jogar(self, historico_oponente):
        if self.estrategia == 'sempre_coopera':
            return 'C'
        elif self.estrategia == 'sempre_trai':
            return 'D'
        elif self.estrategia == 'tit_for_tat':
            return historico_oponente[-1] if historico_oponente else 'C'
        elif self.estrategia == 'aleatorio':
            return random.choice(['C','D'])
        elif self.estrategia == 'rancoroso':
            return 'D' if 'D' in historico_oponente else 'C'

def torneio(agentes, n_rodadas=50):
    for i, a1 in enumerate(agentes):
        for j, a2 in enumerate(agentes):
            if i >= j: continue
            h1, h2 = [], []
            for _ in range(n_rodadas):
                m1 = a1.jogar(h2); m2 = a2.jogar(h1)
                p1, p2 = PAYOFF[(m1,m2)]
                a1.pontos += p1; a2.pontos += p2
                h1.append(m1); h2.append(m2)

random.seed(42)
agentes_dp = [
    AgenteDP("Cooperador",  'sempre_coopera'),
    AgenteDP("Traidor",     'sempre_trai'),
    AgenteDP("Tit-for-Tat", 'tit_for_tat'),
    AgenteDP("Aleatório",   'aleatorio'),
    AgenteDP("Rancoroso",   'rancoroso'),
]
torneio(agentes_dp)
agentes_dp.sort(key=lambda a: -a.pontos)
print(f"{'Estratégia':<15} | {'Pontos':>8}")
print("-"*28)
for a in agentes_dp:
    print(f"{a.nome:<15} | {a.pontos:>8}")
print("\nVencedor:", agentes_dp[0].nome)


### 📝 Exercício Final

Adicione uma estratégia **"generoso tit-for-tat"** que coopera com probabilidade 0.9 após uma traição (em vez de 1.0). Essa pequena aleatoriedade muda o resultado do torneio?

In [ ]:
# ✏️ Adicione a estratégia generosa e execute o torneio novamente:
class AgenteGeneroso(AgenteDP):
    def jogar(self, historico_oponente):
        if not historico_oponente or historico_oponente[-1] == 'C':
            return 'C'
        return 'C' if random.random() < 0.9 else 'D'  # generoso

random.seed(42)
agentes_novo = [
    AgenteDP("Cooperador",  'sempre_coopera'),
    AgenteDP("Traidor",     'sempre_trai'),
    AgenteDP("Tit-for-Tat", 'tit_for_tat'),
    AgenteGeneroso("Generoso", 'tit_for_tat'),
]
torneio(agentes_novo)
agentes_novo.sort(key=lambda a: -a.pontos)
print(f"{'Estratégia':<15} | {'Pontos':>8}")
print("-"*28)
for a in agentes_novo: print(f"{a.nome:<15} | {a.pontos:>8}")
